# 05. Policy Gradients (REINFORCE)

**Nivel:** 🔴 Avanzado  
**Tiempo estimado:** 90 minutos  
**Prerequisitos:** [04. Deep Q-Networks](04-deep-q-networks.ipynb)

## 🎯 Objetivos de Aprendizaje
Al finalizar este notebook, podrás:
- Comprender la diferencia entre métodos basados en valor y en política
- Derivar el teorema del gradiente de política
- Implementar el algoritmo REINFORCE desde cero
- Entender el rol de baselines para reducir varianza
- Aplicar policy gradients a espacios de acciones continuas

## 📚 Motivación

DQN funciona excelentemente para acciones discretas, pero tiene limitaciones:

**1. Solo acciones discretas**: Requiere enumerar todas las acciones posibles
**2. Política determinística**: argmax Q(s,a) siempre elige la misma acción
**3. Optimización indirecta**: Aprende Q-values, no la política directamente

**Policy Gradients** resuelve estos problemas parametrizando y optimizando directamente la política π(a|s; θ).

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import plotly.graph_objects as go
import gymnasium as gym
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.distributions import Categorical
import pandas as pd

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"✅ Librerías importadas | Dispositivo: {device}")

## 📐 Fundamentos Matemáticos

### Política Paramétrica

Parametrizamos la política con una red neuronal:
$$\pi(a|s; \theta) = P(a_t = a | s_t = s; \theta)$$

### Objetivo

Maximizar el retorno esperado:
$$J(\theta) = \mathbb{E}_{\tau \sim \pi_\theta}[R(\tau)]$$

### Teorema del Gradiente de Política

El gradiente del objetivo es:
$$\nabla_\theta J(\theta) = \mathbb{E}\left[\sum_t \nabla_\theta \log \pi_\theta(a_t|s_t) G_t\right]$$

donde $G_t$ es el retorno desde el tiempo $t$.

**Interpretación**: Incrementa probabilidad de acciones que llevaron a buenos retornos.

## 💻 Implementación

In [ ]:
class PolicyNetwork(nn.Module):
    def __init__(self, state_dim, action_dim, hidden_dim=128):
        super().__init__()
        self.fc1 = nn.Linear(state_dim, hidden_dim)
        self.fc2 = nn.Linear(hidden_dim, hidden_dim)
        self.fc3 = nn.Linear(hidden_dim, action_dim)
    
    def forward(self, state):
        x = F.relu(self.fc1(state))
        x = F.relu(self.fc2(x))
        return F.softmax(self.fc3(x), dim=-1)

class REINFORCEAgent:
    def __init__(self, state_dim, action_dim, lr=1e-2, gamma=0.99):
        self.gamma = gamma
        self.policy = PolicyNetwork(state_dim, action_dim).to(device)
        self.optimizer = optim.Adam(self.policy.parameters(), lr=lr)
        self.saved_log_probs = []
        self.rewards = []
        self.history = {'episode': [], 'reward': []}
    
    def select_action(self, state):
        state = torch.FloatTensor(state).unsqueeze(0).to(device)
        probs = self.policy(state)
        m = Categorical(probs)
        action = m.sample()
        self.saved_log_probs.append(m.log_prob(action))
        return action.item()
    
    def update(self):
        returns = []
        R = 0
        for r in reversed(self.rewards):
            R = r + self.gamma * R
            returns.insert(0, R)
        
        returns = torch.tensor(returns).to(device)
        returns = (returns - returns.mean()) / (returns.std() + 1e-9)
        
        policy_loss = []
        for log_prob, R in zip(self.saved_log_probs, returns):
            policy_loss.append(-log_prob * R)
        
        self.optimizer.zero_grad()
        loss = torch.stack(policy_loss).sum()
        loss.backward()
        self.optimizer.step()
        
        self.saved_log_probs = []
        self.rewards = []
    
    def train(self, env, n_episodes=500):
        for ep in range(n_episodes):
            state, _ = env.reset()
            total_reward = 0
            
            for t in range(500):
                action = self.select_action(state)
                next_state, reward, terminated, truncated, _ = env.step(action)
                self.rewards.append(reward)
                total_reward += reward
                
                if terminated or truncated:
                    break
                state = next_state
            
            self.update()
            self.history['episode'].append(ep)
            self.history['reward'].append(total_reward)
            
            if (ep + 1) % 50 == 0:
                avg = np.mean(self.history['reward'][-50:])
                print(f"Episodio {ep+1} | Reward promedio: {avg:.2f}")

print("✅ REINFORCE implementado")

In [ ]:
env = gym.make('CartPole-v1')
agent = REINFORCEAgent(env.observation_space.shape[0], env.action_space.n)

print("🎯 Entrenando REINFORCE en CartPole\n")
agent.train(env, n_episodes=500)

# Visualizar
rewards_smooth = pd.Series(agent.history['reward']).rolling(20, min_periods=1).mean()
fig = go.Figure()
fig.add_trace(go.Scatter(x=agent.history['episode'], y=agent.history['reward'], 
                         mode='lines', name='Reward', opacity=0.3))
fig.add_trace(go.Scatter(x=agent.history['episode'], y=rewards_smooth,
                         mode='lines', name='Media móvil'))
fig.update_layout(title='REINFORCE Training', template='plotly_white')
fig.show()

env.close()

## 🎯 Ejercicios

### 🟢 Ejercicio 1: Baseline
Añade una red de valor como baseline para reducir varianza.

In [ ]:
# TODO: Implementar baseline
pass

### 🟡 Ejercicio 2: Acciones Continuas
Extiende para acciones continuas usando distribución Gaussiana.

In [ ]:
# TODO: Política continua
pass

## 📚 Resumen

### Conceptos Clave
- **Policy Gradient**: Optimiza directamente la política
- **REINFORCE**: Algoritmo básico usando retornos Monte Carlo
- **Baseline**: Reduce varianza
- **Acciones continuas**: Natural con policy gradients

### Próximo Paso
Actor-Critic combina value-based y policy-based methods.

**[Continuar con: 06. Actor-Critic →](06-actor-critic.ipynb)**